# Calculating Power-law scaling in fluctuations of information

In [1]:
import os
import numpy as np
import pandas as pd
from scipy.stats import t

import pyarrow.parquet as pq
from tqdm.notebook import tqdm
import warnings;warnings.filterwarnings('ignore')

In [2]:
CORPUS_NAME = ''

In [3]:
# DATA_PATH = '/Volumes/Data-Repo/shapeoflang3/AF/obs'
DATA_PATH = '../../../updated-data/obs'
NULL_DATA_PATH = '../../../updated-data/null'

OUTPUTS_PATH = 'reports'
if not os.path.exists(OUTPUTS_PATH):
    os.mkdir(OUTPUTS_PATH)

In [4]:
def grab_all_files(PATH, file_type='.parquet'):
    files = [
        [
            os.path.join(root, f) for f in files
            if f.endswith(file_type) and (not f.startswith('.'))
        ]
        for root, _, files in os.walk(PATH)
    ]
    return sum(files, [])

def lang_id(x):
    if 'xling-' in x:
        return x.split('-')[2]
    else:
        return 'eng'

def corpus_id(x):
    if 'xling-' in x:
        return x.split('-')[1]
    else:
        return x.split('-')[0]

In [5]:
fs = [
    f for f in os.listdir(DATA_PATH)
    if ('xling-' not in f)
    and (not f.startswith('.'))
    and (f.endswith('.parquet'))
    and (CORPUS_NAME in f)
]
len(fs)

4462

## Processing individual datasets

In [6]:
# formula = '{} ~ nx + ny + self'.format(COEF_VAR)

In [7]:
COEF_VAR = 'avar'

In [8]:
time_col = 'mtau'
# time_col = 'n'

In [9]:
df_scale, df_regression, k_docs, error_tracker = [], [], dict(), 0

for f in tqdm(fs):
    # print('===][===')
    # print(f.split('/')[-1])

    table = pq.ParquetFile(os.path.join(DATA_PATH, f))

    df = table.read().to_pandas()
    df['self'] = df['x_speaker'] == df['y_speaker']
    df[COEF_VAR] = df[COEF_VAR] #/ df['n']

    # df['mtau'] = (df['min_y_turn_id'] + df['mtau']) - df['x_turn_id']
    # df[COEF_VAR] = np.sqrt(df[COEF_VAR])
    groups = df.groupby(by=['x_speaker', 'y_speaker'])
    # groups = df.groupby(by=['x_turn_id', 'y_speaker'])
    for labels, dfi in groups:
        self_other = dfi['self'].unique()[0]
        # dfi['min_y_turn_id'] = dfi.groupby('x_turn_id')['y_turn_id'].transform('min')

        av = dfi[COEF_VAR].values
        mtau = dfi[time_col].values

        sel = (av > 0) & (mtau > 1)

        try:
            if sel.sum() > 1:

                b, a = np.polyfit(
                    np.log(mtau[sel]),
                    np.log(av[sel]),
                    1
                )

                df_regression += [
                    {
                        'corpus': corpus_id(f),
                        'cid': f.replace('.parquet', ''),
                        'speaker': labels[0],
                        'self': self_other,
                        'a': np.exp(a),
                        'b': b,
                        'k': len(dfi)
                    }
                ]

        except Exception:
            error_tracker += 1

df_regression = pd.DataFrame(df_regression)

  0%|          | 0/4462 [00:00<?, ?it/s]

In [10]:
df_regression.isna().mean()

corpus     0.0
cid        0.0
speaker    0.0
self       0.0
a          0.0
b          0.0
k          0.0
dtype: float64

In [11]:
np.isinf(df_regression['b']).mean()

np.float64(0.0)

In [12]:
# df_regression = df_regression.loc[
#     (df_regression.isna().astype(int).sum(axis=1) == 0)
#     & (np.isinf(df_regression).astype(int).sum(axis=1) == 0)
# ]

In [13]:
df_regression.to_parquet(
    os.path.join(OUTPUTS_PATH,'all.parquet'),
    compression='snappy', engine='fastparquet'
)

## Corpus level analysis of results

In [15]:
split_by = ['corpus', 'self']

In [16]:
df0 = df_regression[split_by+['b']].groupby(by=split_by).agg('mean').reset_index()
df0['std'] = df_regression[split_by+['b']].groupby(by=split_by).agg('std').reset_index()['b'].values

# df0['k'] = [k_docs[corpus][s] for corpus, s in df0[['corpus', 'self']].values]
df0['k'] = df_regression[split_by+['b']].groupby(by=split_by).agg('count').reset_index()['b'].values

df0['se'] = df0['std'] / np.sqrt(df0['k'])

In [17]:
df0['t'] = df0['b'] / df0['se']
df0['p'] = [t.sf(tstat.__abs__(), df=k-1) for tstat, k in df0[['t', 'k']].values]

In [18]:
df0.head(n=50)

,corpus,self,b,std,k,se,t,p
0,CABNC,False,0.551217,0.464442,12291,0.004189,131.578329,0.000000e+00
1,CABNC,True,0.549612,0.603463,4446,0.009050,60.728136,0.000000e+00
2,CANDOR,False,0.786530,0.031083,3304,0.000541,1454.510240,0.000000e+00
3,CANDOR,True,0.846892,0.083651,3304,0.001455,581.936742,0.000000e+00
4,CORAAL,False,0.737195,0.186471,699,0.007053,104.522367,0.000000e+00
5,CORAAL,True,0.779307,0.390184,569,0.016357,47.642548,7.234605e-201
6,DISPEL,False,0.691891,0.180099,38,0.029216,23.681920,2.964941e-24
7,DISPEL,True,0.681831,0.099076,37,0.016288,41.860892,1.964686e-32
8,Frederiksen,False,0.628169,0.078625,4,0.039312,15.978921,2.665070e-04
9,Frederiksen,True,0.701568,0.071331,4,0.035666,19.670702,1.435342e-04


In [19]:
df0.to_csv(
    os.path.join(OUTPUTS_PATH, 'corpus-results.csv'),
    index=False, encoding='utf-8'
)

## Aggregate analysis

In [20]:
split_by = ['self']

In [21]:
df0 = df_regression[split_by+['b']].groupby(by=split_by).agg('mean').reset_index()
df0['std'] = df_regression[split_by+['b']].groupby(by=split_by).agg('std').reset_index()['b'].values
df0['k'] = df_regression[split_by+['b']].groupby(by=split_by).agg('count').reset_index()['b'].values

# df0['k'] = [k_docs[corpus][s] for corpus, s in df0[['corpus', 'self']].values]

df0['se'] = df0['std'] / np.sqrt(df0['k'])

In [22]:
df0['t'] = df0['b'] / df0['se']
df0['p'] = [t.sf(tstat.__abs__(), df=k-1) for tstat, k in df0[['t', 'k']].values]

In [23]:
df0.head(n=20)

,self,b,std,k,se,t,p
0,False,0.556816,0.456155,30174,0.002626,212.039310,0.0
1,True,0.662910,0.499473,10498,0.004875,135.986569,0.0


In [24]:
df0.to_csv(
    os.path.join(OUTPUTS_PATH, 'agg-results.csv'),
    index=False,
    encoding='utf-8'
)